# 第 14 章 · 蒙特卡洛 MC：一局走完，用真实回报的平均估价值

本章回答：不知道环境的 p，还能估出 $v_\pi$ 吗？"每个状态记一个平均回报"具体怎么做？MC 有什么缺点？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/14-monte-carlo.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 一局 | `random_walk_episode(np.random.default_rng(3))` | C → D → E → D → C → D → E，奖励 1 |
| 1000 局 first-visit | `mc_prediction(1000)` | 0.153、0.325、0.495、0.673、0.829 |
| 误差 | `rms(V)` | 0.0084 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `WALK_NAMES`**（来自 `rl_envs.py`）

In [ ]:
WALK_NAMES = ["A", "B", "C", "D", "E"]

**常量 `WALK_TRUE`**（来自 `rl_envs.py`）

In [ ]:
WALK_TRUE = np.array([1, 2, 3, 4, 5]) / 6

**函数 `mc_episode_update`**（来自 `ch14_monte_carlo.py`）：first-visit：每个状态只用它第一次出现之后的回报。alpha=None 时用真平均（1/N）。

In [ ]:
def mc_episode_update(V, episode, alpha=None, counts=None, gamma=1.0):
    """first-visit：每个状态只用它第一次出现之后的回报。alpha=None 时用真平均（1/N）。"""
    G, returns = 0.0, {}
    for s, r, _ in reversed(episode):
        G = r + gamma * G
        returns[s] = G  # 倒着走，最后留下的就是第一次出现时的回报
    for s, G in returns.items():
        if alpha is None:
            counts[s] += 1
            V[s] += (G - V[s]) / counts[s]
        else:
            V[s] += alpha * (G - V[s])
    return V

**函数 `random_walk_episode`**（来自 `rl_envs.py`）：从 C 出发，每步等概率往左或往右；走出左边奖励 0，走出右边奖励 +1。

In [ ]:
def random_walk_episode(rng):
    """从 C 出发，每步等概率往左或往右；走出左边奖励 0，走出右边奖励 +1。
    返回 [(状态, 奖励, 下一个状态或 None)]，状态 0–4 对应 A–E。"""
    s, steps = 2, []
    while True:
        ns = s + (1 if rng.random() < 0.5 else -1)
        if ns < 0:
            steps.append((s, 0.0, None))
            return steps
        if ns > 4:
            steps.append((s, 1.0, None))
            return steps
        steps.append((s, 0.0, ns))
        s = ns

**函数 `mc_prediction`**（来自 `ch14_monte_carlo.py`）：下面要用到的小工具。

In [ ]:
def mc_prediction(n_episodes, alpha=None, seed=0, init=0.5):
    rng = np.random.default_rng(seed)
    V, N = np.full(5, init), np.zeros(5)
    for _ in range(n_episodes):
        mc_episode_update(V, random_walk_episode(rng), alpha, N)
    return V

**函数 `rms`**（来自 `ch14_monte_carlo.py`）：下面要用到的小工具。

In [ ]:
def rms(V):
    return float(np.sqrt(np.mean((V - WALK_TRUE) ** 2)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
ep = random_walk_episode(np.random.default_rng(3))

In [ ]:
print("一局：", " → ".join(WALK_NAMES[s] for s, _, _ in ep), "，最后奖励", ep[-1][1])

In [ ]:
V = mc_prediction(1000)

In [ ]:
print("MC 1000 局（真平均）：", np.round(V, 3), "；真值", np.round(WALK_TRUE, 3))

In [ ]:
print("RMS 误差 =", round(rms(V), 4))

## 3. 代码里才看得到的

用字典 `returns[s] = G` 倒着覆盖，一行就实现了 first-visit：倒着走时最后一次写入 s 的，正是正着数第一次出现的那一步。改成"每次都更新"就是 every-visit。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `rms(mc_prediction(n))`，n = 10 / 100 / 1000 / 10000 → 0.178、0.088、0.0084、0.0031。局数越多越准。
2. `mc_prediction(1000, alpha=0.1)` 改成固定步长，看误差还会不会一直下降。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 不要 p 只要经验，一局局玩、记回报、取平均；
> first-visit 每局只记第一次，倒着算回报最方便；
> 常数 α：V 往 G 挪一步；
> 毛病：要等终局，抖得厉害，没走到的学不到。